# Rubin/LSST — Stable Stars with Known Spectral Type: Rubin Visit Cross-Match

For each photometrically stable star with a **known spectral type** (from the
Simbad master catalogue built in notebook `01_findStarsinSimbad.ipynb`),
this notebook:

1. Loads the master Simbad catalogue and **filters to stars with a defined spectral type**.
2. Loads the Rubin visit table (`visitTable-..._WithTracts.parquet`).
3. **Cross-matches** each star against visits using a cone-search on the visit
   pointing (RA, Dec) — visit footprint approximated by Rubin FoV radius ≈ 1.75°.
4. Saves per-star visit lists (CSV + Parquet) in `data_SIMBAD_02/per_star/`.
5. Produces a **summary table** (visit counts per star × band, sorted by total
   visits descending).


- author : Sylvie Dagoret-Campagne
- affiliation : IJCLab/IN2P3/CNRS, Université Paris-Saclay
- creation : 2026-06-22
- input catalogue : `data_SIMBAD_01/master_stable_stars_V17-22_r1.5deg.csv`
- input visits : `../05_runbindata_visits/data_fromlsst/visitTable-2025041500138-2026053000760_N83426_WithTracts.parquet`


## 1. Imports & configuration

In [ ]:
import os
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm

from astropy.coordinates import SkyCoord
import astropy.units as u

warnings.filterwarnings("ignore")

print(f"numpy   version : {np.__version__}")
print(f"pandas  version : {pd.__version__}")

In [ ]:
try:
    import ipympl  # noqa: F401

    %matplotlib widget
    print("ipympl found → interactive backend")
except ImportError:
    %matplotlib inline
    print("ipympl NOT found → inline")

In [ ]:
# ── Paths ───────────────────────────────────────────────────────────────────────────
# Simbad master catalogue from notebook 01
PATH_SIMBAD_MASTER = "data_SIMBAD_01/master_stable_stars_V17-22_r1.5deg.csv"

# Rubin visit table (WithTracts parquet)
PATH_VISITS = (
    "../05_runbindata_visits/data_fromlsst/visitTable-2025041500138-2026053000760_N83426_WithTracts.parquet"
)

# ── Cross-match radius ─────────────────────────────────────────────────────────────
# Rubin field of view radius ~ 1.75 deg (diameter 3.5 deg).
# We match a star to a visit if the angular distance between the star and
# the visit pointing is smaller than MATCH_RADIUS_DEG.
MATCH_RADIUS_DEG = 1.75  # [adjustable]

# ── Bands to keep ────────────────────────────────────────────────────────────────
BANDS_TO_KEEP = ["u", "g", "r", "i", "z", "y"]  # exclude 'ph', 'unknown'

# ── Output directories ──────────────────────────────────────────────────────────────
NB_TAG = "SIMBAD_02"
DIR_DATA = f"data_{NB_TAG}"
DIR_FIGS = f"figs_{NB_TAG}"
DIR_STARS = os.path.join(DIR_DATA, "per_star")  # one file per star
os.makedirs(DIR_DATA, exist_ok=True)
os.makedirs(DIR_FIGS, exist_ok=True)
os.makedirs(DIR_STARS, exist_ok=True)
print(f"Data root : {os.path.abspath(DIR_DATA)}")
print(f"Per-star  : {os.path.abspath(DIR_STARS)}")
print(f"Figs      : {os.path.abspath(DIR_FIGS)}")

# ── Matplotlib style ───────────────────────────────────────────────────────────────
plt.rcParams.update(
    {
        "figure.dpi": 120,
        "axes.grid": True,
        "grid.alpha": 0.3,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "font.size": 9,
    }
)


def savefig(name: str) -> None:
    for ext in ("pdf", "png"):
        plt.savefig(os.path.join(DIR_FIGS, f"{name}.{ext}"), bbox_inches="tight")
    print(f"  -> saved {name}.{chr(123)}pdf,png{chr(125)}")


print("Configuration done.")

## 2. Load the Simbad master catalogue

We load the master catalogue produced by `01_findStarsinSimbad.ipynb` and
**retain only stars whose spectral type is defined** (non-empty, not `?`, not `unknown`).


In [ ]:
df_simbad = pd.read_csv(PATH_SIMBAD_MASTER)
print(f"Master catalogue shape  : {df_simbad.shape}")
print(f"Columns                 : {list(df_simbad.columns)}")
df_simbad.head(3)

In [ ]:
def has_known_spectral_type(sp) -> bool:
    """Return True when sp is a meaningful spectral-type string."""
    if pd.isna(sp):
        return False
    s = str(sp).strip()
    return s not in ("", "?", "unknown", "--", "nan", "None")


mask_sp = df_simbad["spectral_type"].apply(has_known_spectral_type)
df_stars = df_simbad[mask_sp].copy().reset_index(drop=True)

print(f"Stars with known spectral type : {len(df_stars)} / {len(df_simbad)}")
print("Spectral-type values (top-15)  :")
print(df_stars["spectral_type"].value_counts().head(15).to_string())
df_stars[["simbad_id", "ra_deg", "dec_deg", "V_mag", "spectral_type", "field"]].head(8)

In [ ]:
print("Breakdown by DDF field:")
print(df_stars["field"].value_counts().to_string())

In [ ]:
# ── Extract simple MK temperature class from the first character of spectral_type
def mk_class_from_sptype(sp: str) -> str:
    s = str(sp).strip().upper()
    for cls in ("O", "B", "A", "F", "G", "K", "M", "L", "T", "Y", "W", "C", "S"):
        if s.startswith(cls):
            return cls
    return "?"


df_stars["mk_class_simple"] = df_stars["spectral_type"].apply(mk_class_from_sptype)
print("MK class distribution:")
print(df_stars["mk_class_simple"].value_counts().to_string())

## 3. Load the Rubin visit table

We load the full visit table and **pre-filter to science-quality bands**
(u, g, r, i, z, y) to discard engineering (`ph`, `unknown`) entries.
We also compute the **airmass** from the zenith angle: X = 1 / cos(zenith_angle).


In [ ]:
df_visits_raw = pd.read_parquet(PATH_VISITS)
print(f"Visit table raw shape : {df_visits_raw.shape}")
print(f"Bands present         : {sorted(df_visits_raw['band'].unique())}")
df_visits_raw[["id", "band", "ra", "dec", "zenith_angle", "mjd", "tract", "patch", "target"]].head(3)

In [ ]:
# ── Keep only science bands
df_visits = df_visits_raw[df_visits_raw["band"].isin(BANDS_TO_KEEP)].copy()
df_visits.reset_index(drop=True, inplace=True)

# ── Airmass: plane-parallel approximation X = sec(z) = 1/cos(z_rad)
df_visits["airmass"] = 1.0 / np.cos(np.radians(df_visits["zenith_angle"]))

# ── Rename id → visitId
df_visits.rename(columns={"id": "visitId"}, inplace=True)

# ── Select columns of interest
VISIT_COLS = [
    "visitId",
    "band",
    "ra",
    "dec",
    "mjd",
    "airmass",
    "zenith_angle",
    "tract",
    "patch",
    "patch_str",
    "day_obs",
    "seq_num",
    "time_start",
    "target",
]
df_visits = df_visits[VISIT_COLS]

print(f"Visit table filtered shape : {df_visits.shape}")
print(f"Bands kept                 : {sorted(df_visits['band'].unique())}")
print(f"Airmass range              : [{df_visits['airmass'].min():.3f}, {df_visits['airmass'].max():.3f}]")
df_visits.head(3)

## 4. Build SkyCoord arrays for fast angular cross-match

We use `astropy.coordinates.SkyCoord.separation()` to compute the angular distance
between each star and all visit pointings. The visit SkyCoord array is built **once**
before the star loop for efficiency.


In [ ]:
# SkyCoord for ALL visits (built once)
visit_coords = SkyCoord(
    ra=df_visits["ra"].values * u.deg,
    dec=df_visits["dec"].values * u.deg,
    frame="icrs",
)
print(f"Visit SkyCoord array : {len(visit_coords)} entries")

# SkyCoord for ALL stars (used for sky-distribution plot later)
star_coords = SkyCoord(
    ra=df_stars["ra_deg"].values * u.deg,
    dec=df_stars["dec_deg"].values * u.deg,
    frame="icrs",
)
print(f"Star SkyCoord array  : {len(star_coords)} entries")

## 5. Per-star visit cross-match and saving

For each star we:
- Select visits within `MATCH_RADIUS_DEG` of the star's sky position.
- Annotate the sub-table with star metadata (`simbad_id`, `spectral_type`, `V_mag`, `field`).
- Save CSV + Parquet in `data_SIMBAD_02/per_star/<star_id>.{csv,parquet}`.
- Accumulate per-band counts for the summary table.


In [ ]:
match_radius = MATCH_RADIUS_DEG * u.deg
summary_rows = []

for idx, star in df_stars.iterrows():
    star_coord = SkyCoord(ra=star["ra_deg"] * u.deg, dec=star["dec_deg"] * u.deg, frame="icrs")

    sep = star_coord.separation(visit_coords)
    in_fov = sep < match_radius
    df_match = df_visits[in_fov].copy()

    row = {
        "simbad_id": star["simbad_id"],
        "spectral_type": star["spectral_type"],
        "mk_class_simple": star.get("mk_class_simple", "?"),
        "V_mag": star["V_mag"],
        "ra_deg": star["ra_deg"],
        "dec_deg": star["dec_deg"],
        "field": star["field"],
        "n_visits_total": len(df_match),
    }
    for b in BANDS_TO_KEEP:
        row[f"n_{b}"] = 0

    if not df_match.empty:
        df_match["simbad_id"] = star["simbad_id"]
        df_match["spectral_type"] = star["spectral_type"]
        df_match["V_mag"] = star["V_mag"]
        df_match["field"] = star["field"]

        star_id_clean = star["simbad_id"].replace(" ", "_").replace("/", "-").replace(":", "-")

        df_match.to_csv(os.path.join(DIR_STARS, f"{star_id_clean}.csv"), index=False)
        df_match.to_parquet(os.path.join(DIR_STARS, f"{star_id_clean}.parquet"), index=False)

        band_counts = df_match["band"].value_counts()
        for b in BANDS_TO_KEEP:
            row[f"n_{b}"] = int(band_counts.get(b, 0))

    summary_rows.append(row)

    if (idx + 1) % 20 == 0 or (idx + 1) == len(df_stars):
        print(f"  [{idx + 1:4d}/{len(df_stars)}] {star['simbad_id']:40s}  → {len(df_match):5d} visits")

print(f"\nDone. Per-star files saved to: {os.path.abspath(DIR_STARS)}")

## 6. Summary table — visit counts per star × band

One row per star, sorted by `n_visits_total` descending.
Saved as CSV and Parquet in `data_SIMBAD_02/`.


In [ ]:
df_summary = pd.DataFrame(summary_rows)
df_summary.sort_values("n_visits_total", ascending=False, inplace=True)
df_summary.reset_index(drop=True, inplace=True)

col_order = [
    "simbad_id",
    "spectral_type",
    "mk_class_simple",
    "V_mag",
    "ra_deg",
    "dec_deg",
    "field",
    "n_visits_total",
] + [f"n_{b}" for b in BANDS_TO_KEEP]
df_summary = df_summary[col_order]

print(f"Summary table shape : {df_summary.shape}")
df_summary.head(15)

In [ ]:
out_csv = os.path.join(DIR_DATA, "summary_visit_counts_per_star.csv")
out_parquet = os.path.join(DIR_DATA, "summary_visit_counts_per_star.parquet")
df_summary.to_csv(out_csv, index=False)
df_summary.to_parquet(out_parquet, index=False)
print(f"Saved: {out_csv}")
print(f"Saved: {out_parquet}")

In [ ]:
print("=== Visit-count statistics (total visits per star) ===")
print(df_summary["n_visits_total"].describe().to_string())
print()
print("=== Stars with >= 100 visits ===")
print(
    df_summary[df_summary["n_visits_total"] >= 100][
        ["simbad_id", "spectral_type", "field", "n_visits_total"]
    ].to_string()
)

## 7. Visualisations

### 7.1  Visit-count heatmap (stars × bands)


In [ ]:
df_plot = df_summary[df_summary["n_visits_total"] > 0].head(40)
band_cols = [f"n_{b}" for b in BANDS_TO_KEEP]
heat_data = df_plot[band_cols].values.astype(float)
ylabels = [f"{row.simbad_id}  [{row.spectral_type}]" for _, row in df_plot.iterrows()]

fig, ax = plt.subplots(figsize=(10, max(4, len(df_plot) * 0.35)))
im = ax.imshow(heat_data, aspect="auto", cmap="YlOrRd")
ax.set_xticks(range(len(BANDS_TO_KEEP)))
ax.set_xticklabels(BANDS_TO_KEEP, fontsize=10)
ax.set_yticks(range(len(ylabels)))
ax.set_yticklabels(ylabels, fontsize=7)
ax.set_xlabel("Band")
ax.set_title(f"Visit counts per star × band  (top-{len(df_plot)} by n_visits_total)")
plt.colorbar(im, ax=ax, label="N visits")
plt.tight_layout()
savefig("heatmap_visits_star_band")
plt.show()

### 7.2  Total visits distribution by MK spectral class

In [ ]:
mk_classes = sorted(df_summary["mk_class_simple"].unique())
cmap_mk = cm.get_cmap("tab10", len(mk_classes))

fig, ax = plt.subplots(figsize=(8, 4))
for i, cls in enumerate(mk_classes):
    sub = df_summary[df_summary["mk_class_simple"] == cls]["n_visits_total"]
    if sub.empty:
        continue
    ax.hist(sub, bins=30, alpha=0.55, label=f"{cls} ({len(sub)})", color=cmap_mk(i))
ax.set_xlabel("N visits total")
ax.set_ylabel("N stars")
ax.set_title("Total visit count distribution by MK class")
ax.legend(fontsize=8, ncol=3)
plt.tight_layout()
savefig("hist_visits_by_mk_class")
plt.show()

### 7.3  Sky distribution coloured by total visit count

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
sc = ax.scatter(
    df_summary["ra_deg"],
    df_summary["dec_deg"],
    c=df_summary["n_visits_total"],
    cmap="plasma",
    s=20,
    alpha=0.8,
    edgecolors="none",
)
plt.colorbar(sc, ax=ax, label="N visits total")
ax.set_xlabel("RA (deg)")
ax.set_ylabel("Dec (deg)")
ax.set_title("Sky distribution of stable stars with known spectral type\n(colour = N Rubin visits)")
ax.invert_xaxis()
plt.tight_layout()
savefig("sky_distribution_visit_count")
plt.show()

### 7.4  Per-band visit counts vs. V magnitude

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(13, 7), sharex=True)
axes = axes.flatten()

for i, band in enumerate(BANDS_TO_KEEP):
    ax = axes[i]
    col = f"n_{band}"
    sub = df_summary[df_summary[col] > 0]
    ax.scatter(sub["V_mag"], sub[col], s=12, alpha=0.6, edgecolors="none")
    ax.set_title(f"Band {band}  ({len(sub)} stars)")
    ax.set_xlabel("V mag")
    ax.set_ylabel(f"N visits ({band})")

plt.suptitle("Per-band visit count vs. V magnitude", y=1.01, fontsize=11)
plt.tight_layout()
savefig("visits_vs_vmag_per_band")
plt.show()

## 8. Example: visit list for the best-observed star

Display the full visit list for the star with the highest total visit count,
and plot its MJD timeline coloured by band.


In [ ]:
best_star = df_summary.iloc[0]
print("Best-observed star:")
print(f"  simbad_id     : {best_star['simbad_id']}")
print(f"  spectral_type : {best_star['spectral_type']}")
print(f"  V_mag         : {best_star['V_mag']:.2f}")
print(f"  field         : {best_star['field']}")
print(f"  n_visits_total: {best_star['n_visits_total']}")
for b in BANDS_TO_KEEP:
    print(f"    n_{b}: {int(best_star[f'n_{b}'])}")

In [ ]:
star_id_clean = best_star["simbad_id"].replace(" ", "_").replace("/", "-").replace(":", "-")
path_best = os.path.join(DIR_STARS, f"{star_id_clean}.parquet")

df_best = pd.read_parquet(path_best)
print(f"Shape of visit list: {df_best.shape}")
df_best[["visitId", "band", "mjd", "airmass", "ra", "dec", "tract", "patch_str", "time_start"]].head(10)

In [ ]:
band_colors = {
    "u": "navy",
    "g": "forestgreen",
    "r": "crimson",
    "i": "darkorange",
    "z": "purple",
    "y": "saddlebrown",
}

fig, ax = plt.subplots(figsize=(12, 3))
for band, grp in df_best.groupby("band"):
    ax.scatter(
        grp["mjd"],
        grp["airmass"],
        label=band,
        color=band_colors.get(band, "grey"),
        s=15,
        alpha=0.7,
        edgecolors="none",
    )
ax.set_xlabel("MJD")
ax.set_ylabel("Airmass")
ax.set_title(
    f"Visit timeline for {best_star['simbad_id']}  [{best_star['spectral_type']}]  V={best_star['V_mag']:.1f}"
)
ax.legend(title="band", ncol=6, fontsize=8)
plt.tight_layout()
savefig(f"timeline_{star_id_clean}")
plt.show()

## 9. Printed summary table (top 30 stars)

Final pretty-printed summary sorted by total visit count.


In [ ]:
cols_display = ["simbad_id", "spectral_type", "mk_class_simple", "V_mag", "field", "n_visits_total"] + [
    f"n_{b}" for b in BANDS_TO_KEEP
]

pd.set_option("display.max_colwidth", 35)
pd.set_option("display.max_rows", 30)
df_summary[cols_display].head(30)

In [ ]:
print("=" * 62)
print("Summary")
print("=" * 62)
print(f"  Total stable stars with known spectral type : {len(df_stars)}")
print(f"  Stars with >= 1 Rubin visit                : {(df_summary['n_visits_total'] > 0).sum()}")
print(f"  Stars with   0 Rubin visits                : {(df_summary['n_visits_total'] == 0).sum()}")
print(f"  Per-star files saved in                    : {os.path.abspath(DIR_STARS)}")
print(f"  Summary table saved in                     : {os.path.abspath(DIR_DATA)}")